# Sandbox Analysis: Natural Language Bioeconomy Exploration

Welcome to the AI-driven data sandbox for the `ca-biositing` project. This notebook uses **PandasAI 3.0+** and the **CBORG API** (Gemini models) to allow you to explore geospatial bioeconomy data using natural language.

### Capabilities:
- **Natural Language Querying**: Ask questions about the data in plain English.
- **Interactive Visualizations**: Generate Plotly charts for zooming and hovering.
- **Local DB Integration**: Queries the live `ca_biositing` PostgreSQL instance.

## 1. Setup and Environment

We initialize our connection to the CBORG LLM gateway and apply a **Global Patch** to PandasAI to ensure all interactive charts render directly in the notebook.

In [1]:
import sys
import os
from pathlib import Path

print(f"Python Executable: {sys.executable}")
print(f"Current Working Directory: {os.getcwd()}")

# Force discovery of local modules in analysis/ai_exploration
def find_setup_dir():
    # Try relative to root, then relative to current
    for p in [Path("analysis/ai_exploration"), Path(".")]:
        if (p / "sandbox_setup.py").exists():
            return p.resolve()
    return None

setup_dir = find_setup_dir()
if setup_dir:
    if str(setup_dir) not in sys.path:
        sys.path.insert(0, str(setup_dir))
        print(f"Added {setup_dir} to sys.path")
else:
    print("WARNING: Could not find sandbox_setup.py directory automatically.")

try:
    from sandbox_setup import init_sandbox, get_agent_with_metadata
    print("SUCCESS: sandbox_setup imported")
except ImportError as e:
    print(f"FAILURE: {e}")
    print(f"Path: {sys.path}")
    raise

llm, engine = init_sandbox(model_name="gemini-2.0-flash")

Python Executable: /Users/pjsmitty301/ca-biositing/analysis/ai_exploration/.pixi/envs/default/bin/python
Current Working Directory: /Users/pjsmitty301/ca-biositing
Added /Users/pjsmitty301/ca-biositing/analysis/ai_exploration to sys.path
SUCCESS: sandbox_setup imported
Initialized AI Sandbox with model: gemini-2.0-flash
Connected to Database: biocirv_db


## 3. Loading Data and Initializing Agent

We load all relevant views into the agent.

In [2]:
views = ["analysis_data_view", "usda_census_view", "analysis_average_view"]

# Initialize the Agent with SQL Connectors and schema-aware metadata
agent = get_agent_with_metadata(llm, engine, views)

print(f"\nAgent initialized with {len(views)} views and custom SQL skill.")

- Loaded analysis_data_view (5000 rows)
- Loaded usda_census_view (1419 rows)
- Loaded analysis_average_view (773 rows)

Agent initialized with 3 views and custom SQL skill.


## 4. Interactive Queries

Ask for an interactive chart below. The global patch ensures it renders as interactive HTML in the cell output.

In [3]:
agent.chat("What are the top 5 most common resources in the analysis_data_view?")

Executing SQL: 
SELECT resource, COUNT(*) AS count
FROM analysis_data_view
GROUP BY resource
ORDER BY count DESC
LIMIT 5;



DataFrameResponse(type='dataframe', value=        resource  count
0           None   1056
1  tomato pomace    902
2   almond hulls    709
3  almond shells    665
4   grape pomace    615)

### A. Interactive Visualization
Visualize the energy content distribution with an interactive Plotly bar chart.

In [ ]:
agent.chat("Can you also provide the unit with the last chart you generated?")

In [10]:
agent.chat("Can you calculate the spearman correlation between all elements in icp_analysis in the analysis_data_view. Please produce a dataframe with the result.")

Executing SQL: 
SELECT 
    parameter,
    record_id,
    value
FROM analysis_data_view
WHERE record_type = 'icp analysis'

{'type': 'plot', 'value': 'exports/charts/temp_chart_8aa5a6ac-c2b7-4ca2-9c5f-b61f71e20d70.png'}


ChartResponse(type='chart', value='exports/charts/temp_chart_8aa5a6ac-c2b7-4ca2-9c5f-b61f71e20d70.png')

In [9]:
agent.last_code_executed

'import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nsql_query = """\nSELECT \n    parameter,\n    record_id,\n    value\nFROM analysis_data_view\nWHERE record_type = \'icp analysis\'\n"""\ndf = execute_sql_query(sql_query)\nif df is not None and not df.empty:\n    try:\n        pivot_table = df.pivot_table(values=\'value\', index=\'record_id\', columns=\'parameter\')\n        correlation_matrix = pivot_table.corr(method=\'spearman\')\n        if correlation_matrix.isnull().values.all():\n            result = {\'type\': \'answer\', \'value\': \'All correlations are NaN. This likely indicates that there is only one unique value for each parameter, or that there is no variance in the data, preventing the calculation of meaningful correlations.\'}\n        else:\n            plt.figure(figsize=(10, 8))\n            sns.heatmap(correlation_matrix, annot=True, cmap=\'coolwarm\', fmt=\'.2f\')\n            plt.title(\'Spearman Rank Correlation Matr

### B. Scatter Plot
Visualize relationship between two variables from the average view.

In [4]:
agent.chat("Can you make an interactive plot of average xylan and glucan content. Label all almond associated resources with a different color and label the axes with the correct units?")

Executing SQL: 
SELECT 
    resource,
    AVG(CASE WHEN parameter = 'xylan' THEN average_value ELSE NULL END) AS avg_xylan,
    AVG(CASE WHEN parameter = 'glucan' THEN average_value ELSE NULL END) AS avg_glucan
FROM analysis_average_view
WHERE parameter IN ('xylan', 'glucan')
GROUP BY resource



ChartResponse(type='chart', value='scatter_plot.html')

In [11]:
agent.chat("Can you make the plot interactive and label all almond assosciated resources?")

Executing SQL: 
SELECT 
    resource,
    AVG(CASE WHEN parameter LIKE 'lignin%' THEN average_value ELSE NULL END) AS avg_lignin,
    AVG(CASE WHEN parameter = 'volatile solids' THEN average_value ELSE NULL END) AS avg_volatile_solids
FROM 
    analysis_average_view
WHERE parameter LIKE 'lignin%' OR parameter = 'volatile solids'
GROUP BY resource
HAVING AVG(CASE WHEN parameter LIKE 'lignin%' THEN average_value ELSE NULL END) IS NOT NULL AND AVG(CASE WHEN parameter = 'volatile solids' THEN average_value ELSE NULL END) IS NOT NULL



ChartResponse(type='chart', value='exports/charts/temp_chart_interactive.html')

### C. Multi-DataFrame Analysis
Correlate production data with USDA census data.

In [ ]:
agent.chat("Summarize how the county-level production in the analysis data relates to the census data in the USDA view.")